# Stjørdal Dataset — Thesis Presentation
This notebook generates thesis-ready summary tables and figures for the Stjørdal (Skatval) dataset. It enumerates available radii, computes per-session summaries, event balances, location and label summary tables, and saves figures/tables under `outputs/dataset_statistics_thesis/`.

In [7]:
from pathlib import Path
import sys, importlib.util
import pandas as pd
from IPython.display import display
# Load helper script as module to reuse dataset-statistics functions
# When executed via nbconvert the notebook's cwd is the notebooks/ folder,
# so set proj_root to the parent folder (project root).
proj_root = Path('..').resolve()
script_path = proj_root / 'scripts' / '03_dataset_statistics.py'
spec = importlib.util.spec_from_file_location('ds_stats', str(script_path))
ds_stats = importlib.util.module_from_spec(spec)
import sys as _sys
_sys.modules['ds_stats'] = ds_stats
spec.loader.exec_module(ds_stats)
print('Loaded dataset utilities from', script_path)


Loaded dataset utilities from C:\Users\imborhau\Documents\sound-event-detection-aircrafts\scripts\03_dataset_statistics.py


In [8]:
# Configuration - adjust if neededscripts/03_dataset_statistics.py
DATASET_ROOT = Path(proj_root,'dataset/Skatval').resolve()
OUTPUT_ROOT = Path('outputs/dataset_statistics_thesis').resolve()
print('Dataset root:', DATASET_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Sessions to include (uses the defaults from the script)
SESSION_NAMES = ds_stats.SESSION_NAMES
# Discover all radii present in GT filenames (single discovery)
gt_files = sorted(DATASET_ROOT.rglob('*_AUTOSAVE_sphere_*KM.csv'))
radii = sorted({ds_stats.parse_gt_name(p)[2] for p in gt_files})
print('Found radii:', radii)

Dataset root: C:\Users\imborhau\Documents\sound-event-detection-aircrafts\dataset\Skatval
Found radii: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]


In [9]:
# Iterate radii and produce per-radius tables + figures
all_summary = []
all_balance = []
all_events = []
for radius in radii:
    print(f'Processing radius {radius} KM...')
    session_data = ds_stats.discover_session_data(DATASET_ROOT, SESSION_NAMES, radius)
    events_df = ds_stats.build_events_dataframe(session_data)
    summary_df = ds_stats.make_summary_table(session_data, events_df, ds_stats.params.PATCH_HOP_SECONDS)
    balance_df = ds_stats.make_event_balance_table(session_data, events_df)
    # Save per-radius outputs
    out_dir = OUTPUT_ROOT / f"radius_{str(radius).replace('.','_')}KM"
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'tables').mkdir(exist_ok=True)
    summary_df.to_csv(out_dir / 'tables' / 'dataset_summary_by_session.csv', index=False)
    balance_df.to_csv(out_dir / 'tables' / 'event_balance_by_session.csv', index=False)
    events_df.to_csv(out_dir / 'tables' / 'all_positive_events.csv', index=False)
    # Generate figures (uses the script's plotting helpers)
    ds_stats.create_figures(summary_df, events_df, out_dir, event_label='Aircraft', radius_km=radius)
    print('Saved outputs to', out_dir)
    # collect for combined tables
    if not summary_df.empty:
        s = summary_df.copy(); s['radius_km'] = radius; all_summary.append(s)
    if not balance_df.empty:
        b = balance_df.copy(); b['radius_km'] = radius; all_balance.append(b)
    if not events_df.empty:
        e = events_df.copy(); e['radius_km'] = radius; all_events.append(e)

# Concatenate combined tables and save
if all_summary:
    combined_summary = pd.concat(all_summary, ignore_index=True)
    combined_summary.to_csv(OUTPUT_ROOT / 'combined_summary_all_radii.csv', index=False)
    display(combined_summary)
else:
    print('No summary rows collected')

if all_balance:
    combined_balance = pd.concat(all_balance, ignore_index=True)
    combined_balance.to_csv(OUTPUT_ROOT / 'combined_balance_all_radii.csv', index=False)
    display(combined_balance)

if all_events:
    combined_events = pd.concat(all_events, ignore_index=True)
    combined_events.to_csv(OUTPUT_ROOT / 'combined_all_positive_events_all_radii.csv', index=False)
    print('Saved combined events table')


Processing radius 1.0 KM...
Saved outputs to C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\radius_1_0KM
Processing radius 2.0 KM...
Saved outputs to C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\radius_2_0KM
Processing radius 3.0 KM...
Saved outputs to C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\radius_3_0KM
Processing radius 4.0 KM...
Saved outputs to C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\radius_4_0KM
Processing radius 5.0 KM...
Saved outputs to C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\radius_5_0KM
Processing radius 6.0 KM...
Saved outputs to C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\radius_6_0KM
Processing radius 7.0 KM...


,session,num_wav_files,recorded_hours,total_labeled_events,positive_samples,negative_samples,neg_to_pos_ratio,mean_event_duration_s,median_event_duration_s,min_event_duration_s,max_event_duration_s,radius_km
0,300925,1,1.950394,21,1266,13362,10.554502,28.937143,28.80,19.20,39.36,1.0
1,280126,3,6.593542,1,62,49390,796.612903,29.760000,29.76,29.76,29.76,1.0
2,230226,3,6.045139,0,0,45339,inf,0.000000,0.00,0.00,0.00,1.0
3,030326,2,4.483287,0,0,33625,inf,0.000000,0.00,0.00,0.00,1.0
4,260326_part1,3,10.410208,2,80,77997,974.962500,19.200000,19.20,19.20,19.20,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
85,280126,3,6.593542,51,25256,24196,0.958030,478.983529,438.72,24.96,1818.24,15.0
86,230226,3,6.045139,66,34202,11137,0.325624,439.243636,488.64,19.20,888.96,15.0
87,030326,2,4.483287,47,19038,14587,0.766204,342.720000,229.44,8.64,819.84,15.0
88,260326_part1,3,10.410208,135,44202,33875,0.766368,184.952889,168.96,18.24,648.96,15.0


,session,positive_duration_s,negative_duration_s,positive_percent,negative_percent,radius_km
0,300925,607.68,6413.74,8.65,91.35,1.0
1,280126,29.76,23706.99,0.13,99.87,1.0
2,230226,0.00,21762.50,0.00,100.00,1.0
3,030326,0.00,16139.83,0.00,100.00,1.0
4,260326_part1,38.40,37438.35,0.10,99.90,1.0
...,...,...,...,...,...,...
85,280126,12122.88,11613.87,51.07,48.93,15.0
86,230226,16416.96,5345.54,75.44,24.56,15.0
87,030326,9138.24,7001.59,56.62,43.38,15.0
88,260326_part1,21216.96,16259.79,56.61,43.39,15.0


Saved combined events table


In [10]:
# Location summary table (auto-filled with Location IDs; other fields left as TBD for manual editing)
locs = ds_stats.LOCATION_ORDER
loc_rows = []
for loc in locs:
    loc_rows.append({
        'location_id': loc,
        'municipality': 'TBD',
        'environment_type': 'TBD',
        'coordinates': 'TBD',
        'elevation_m': 'TBD',
        'airport_proximity': 'TBD',
        'typical_aircraft_pattern': 'TBD',
        'main_background_noise_sources': 'TBD',
    })
location_df = pd.DataFrame(loc_rows)
location_df.to_csv(OUTPUT_ROOT / 'location_summary.csv', index=False)
display(location_df)

,location_id,municipality,environment_type,coordinates,elevation_m,airport_proximity,typical_aircraft_pattern,main_background_noise_sources
0,loc_1,TBD,TBD,TBD,TBD,TBD,TBD,TBD
1,loc_2,TBD,TBD,TBD,TBD,TBD,TBD,TBD
2,loc_3,TBD,TBD,TBD,TBD,TBD,TBD,TBD
3,gardemoen,TBD,TBD,TBD,TBD,TBD,TBD,TBD


**Label / Annotation Summary**
The notebook will generate and save a label-summary CSV below; defaults are used unless you edit them manually.

In [11]:
label_summary = {
    'annotation_source': 'FlightRadar24',
    'distance_method': 'ECEF Euclidean distance',
    'positive_label_definition': 'Intervals overlapping aircraft event (GT class positive)',
    'negative_label_definition': 'No overlap with aircraft event',
    'patch_duration_s': ds_stats.params.PATCH_SECONDS if hasattr(ds_stats.params, 'PATCH_SECONDS') else 'TBD',
    'patch_hop_s': ds_stats.params.PATCH_HOP_SECONDS if hasattr(ds_stats.params, 'PATCH_HOP_SECONDS') else 'TBD',
    'altitude_filter': 'As specified in annotation pipeline (if any)',
    'radii_reported': radii,
    'ambiguous_intervals_excluded': True,
}
label_df = pd.DataFrame([label_summary])
label_df.to_csv(OUTPUT_ROOT / 'label_annotation_summary.csv', index=False)
display(label_df.T)

,0
annotation_source,FlightRadar24
distance_method,ECEF Euclidean distance
positive_label_definition,Intervals overlapping aircraft event (GT class...
negative_label_definition,No overlap with aircraft event
patch_duration_s,TBD
patch_hop_s,0.48
altitude_filter,As specified in annotation pipeline (if any)
radii_reported,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, ..."
ambiguous_intervals_excluded,True


In [12]:
# Per-radius figures: percentage of positive events per session for radii 1..9 KM
import matplotlib.pyplot as plt
from pathlib import Path
# Load balance table (combined) or reconstruct from per-radius tables
combined_path = OUTPUT_ROOT / 'combined_balance_all_radii.csv'
if combined_path.exists():
    balance = pd.read_csv(combined_path)
else:
    rows = []
    for p in (OUTPUT_ROOT).glob('radius_*'):
        t = Path(p) / 'tables' / 'event_balance_by_session.csv'
        if t.exists():
            df = pd.read_csv(t)
            try:
                df['radius_km'] = float(p.name.split('_')[1].replace('_','.'))
            except Exception:
                df['radius_km'] = None
            rows.append(df)
    if rows:
        balance = pd.concat(rows, ignore_index=True)
    else:
        raise FileNotFoundError('No combined or per-radius balance tables found under OUTPUT_ROOT')
# Compute positive percent if needed (reuse heuristic from earlier cell)
def ensure_positive_percent(df):
    if 'positive_percent' in df.columns:
        return df
    if {'positive_duration','duration'}.issubset(df.columns):
        df['positive_percent'] = 100 * df['positive_duration'] / df['duration']
        return df
    if {'positive_events','total_events'}.issubset(df.columns):
        df['positive_percent'] = 100 * df['positive_events'] / df['total_events']
        return df
    pos_col = next((c for c in df.columns if 'positive' in c.lower()), None)
    tot_col = next((c for c in df.columns if any(k in c.lower() for k in ('total','duration','count','n_'))), None)
    if pos_col and tot_col:
        df['positive_percent'] = 100 * df[pos_col] / df[tot_col]
        return df
    raise ValueError('Could not determine positive and total columns to compute positive percent')
balance = ensure_positive_percent(balance)
# Determine session column
session_col = None
for c in ('session_name','session'):
    if c in balance.columns:
        session_col = c; break
if session_col is None:
    session_col = next((c for c in balance.columns if 'session' in c.lower()), None)
if session_col is None:
    raise ValueError('Could not find session column in balance table')
# Prepare figure directory
fig_dir = OUTPUT_ROOT / 'figures' / 'per_radius_positive_percent'
fig_dir.mkdir(parents=True, exist_ok=True)
# Loop radii 1..9 and plot
for r in range(1,10):
    r_f = float(r)
    sub = balance[balance['radius_km'] == r_f] if 'radius_km' in balance.columns else balance[balance['radius'] == r_f]
    if sub.empty:
        print(f'No data for radius {r} KM; skipping')
        continue
    sub = sub.set_index(session_col).sort_index()
    values = sub['positive_percent']
    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(values))))
    values.plot(kind='bar', ax=ax, color='C0')
    ax.set_ylim(0, min(100, max(10, values.max() * 1.1)))
    ax.set_ylabel('Positive events (%)')
    ax.set_title(f'Positive events (%) per session — radius {r} KM')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out_png = fig_dir / f'radius_{r}_positive_percent_by_session.png'
    out_pdf = fig_dir / f'radius_{r}_positive_percent_by_session.pdf'
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_pdf)
    print('Saved', out_png)
    display(values.to_frame(name='positive_percent'))
    plt.close(fig)

Saved C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\figures\per_radius_positive_percent\radius_1_positive_percent_by_session.png


,positive_percent
session,
030326,0.00
230226,0.00
260326_part1,0.10
260326_part2,0.00
280126,0.13
300925,8.65


Saved C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\figures\per_radius_positive_percent\radius_2_positive_percent_by_session.png


,positive_percent
session,
030326,0.00
230226,1.48
260326_part1,0.93
260326_part2,1.33
280126,1.36
300925,17.97


Saved C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\figures\per_radius_positive_percent\radius_3_positive_percent_by_session.png


...

,positive_percent
session,
030326,40.76
230226,50.08
260326_part1,23.04
260326_part2,24.69
280126,35.63
300925,77.67


Saved C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\figures\per_radius_positive_percent\radius_8_positive_percent_by_session.png


,positive_percent
session,
030326,44.44
230226,68.30
260326_part1,29.19
260326_part2,29.85
280126,46.22
300925,80.26


Saved C:\Users\imborhau\Documents\sound-event-detection-aircrafts\notebooks\outputs\dataset_statistics_thesis\figures\per_radius_positive_percent\radius_9_positive_percent_by_session.png


,positive_percent
session,
030326,47.27
230226,69.46
260326_part1,34.40
260326_part2,33.48
280126,46.94
300925,82.98


\begin{table}[htbp]
\centering
\caption{Overview of the Stjørdal dataset.}
\label{tab:stjordal-overview}
\begin{tabular}{lcccc}
\toprule
Session & Date & Locations & Duration [h] & Time span [h] \\
\midrule
1 & 28.01.26 & 3 & 6.59 & 2.20 \\
2 & 23.02.26 & 3 & 6.05 & 2.02 \\
3 & 03.03.26 & 2 & 4.48 & 2.24 \\
4 & 26.03.26 & 3 & 20.88 & 6.96 \\
5 & 30.09.25 & 1 & 1.95 & 1.95 \\
\midrule
\textbf{Total} &  & 12 & 39.97 & 15.36 \\
\bottomrule
\end{tabular}
\end{table}

How the table values are calculated:
- \textbf{Locations}: the number of microphone locations present in that session folder. Session 1, 2, and 4 have three locations; Session 3 has two; Session 5 has one.
- \textbf{Duration [h]}: the total raw audio duration in hours, summed over all `.wav` files in the session. This is computed from each file's frame count divided by its sample rate.
- \textbf{Time span [h]}: the recording window covered by the session, from the first timestamp in the raw position log to the last timestamp. For Session 4, the two parts are combined before computing the span.


**Notes / Reminders**
- `Duration` values are total raw audio duration (wav files).
- Positive/Negative durations are computed from annotation intervals at patch hop resolution; patch sampling used is `keras_yamnet.params.PATCH_HOP_SECONDS`.
- `260326_part1` and `260326_part2` were merged as requested.
- Location table contains placeholders (TBD) for fields you indicated should be manually verified.